# DOA Pipeline — `static_10m_000.wav`

Streams `data/static_10m_000.wav` through the full pipeline:
`WavLoader → StftProcessor → DoaProcessor`
and prints every chunk at each stage.

**File:** 8 channels, 44100 Hz, ~1.08 s  
**DOA mic array:** outer ring (channels 1–4, indices 0–3)

In [2]:
import sys
sys.path.insert(0, '..')  # make logic/ importable from notebooks/

import numpy as np
from logic.wav_loader import WavLoader
from logic.stft_processor import StftProcessor, StftChunk
from logic.doa_processor import DoaProcessor

In [9]:
# --- Config ---
WAV_FILE      = '../data/static_10m_000.wav'
SAMPLE_RATE   = 44100
CHUNK_SIZE    = 8192
NPERSEG       = 512
NOVERLAP      = 256
OUTER_INDICES = slice(0, 4)  # channels 1-4 in the WAV file

# Outer ring (channels 1–4): 34 cm square, shape (3, 4)
L_outer = np.array([
    [-0.17,  0.17, -0.17,  0.17],   # x
    [-0.17, -0.17,  0.17,  0.17],   # y
    [ 0.00,  0.00,  0.00,  0.00],   # z
])

In [10]:
# --- Pipeline ---
wav_loader     = WavLoader(WAV_FILE, chunk_size=CHUNK_SIZE)
stft_processor = StftProcessor(nperseg=NPERSEG, noverlap=NOVERLAP)
doa_processor  = DoaProcessor(mic_locs=L_outer, sampling_rate=SAMPLE_RATE, nfft=NPERSEG)

In [11]:
SEP = '─' * 68

def audio_stream():
    for chunk in wav_loader.stream():
        print(f"{chunk.timestamp:7.3f}s  AudioChunk   shape={chunk.data.shape}")
        yield chunk

def stft_stream(audio_chunks):
    for chunk in stft_processor.process(audio_chunks):
        print(f"{chunk.timestamp:7.3f}s  StftChunk    "
              f"magnitudes={chunk.magnitudes.shape}  "
              f"freqs={chunk.freqs[0]:.0f}–{chunk.freqs[-1]:.0f} Hz  "
              f"frames={chunk.magnitudes.shape[2]}")
        # Select outer-ring channels only before passing to DOA
        yield StftChunk(
            freqs=chunk.freqs,
            times=chunk.times,
            magnitudes=chunk.magnitudes[OUTER_INDICES],
            sampling_rate=chunk.sampling_rate,
            timestamp=chunk.timestamp,
        )

print(SEP)
print(f"{'timestamp':<10} {'type':<14} {'details'}")
print(SEP)

for doa_chunk in doa_processor.process(stft_stream(audio_stream())):
    print(f"{doa_chunk.timestamp:7.3f}s  DoaChunk     azimuth={doa_chunk.azimuth_deg:.1f}°")

print(SEP)

────────────────────────────────────────────────────────────────────
timestamp  type           details
────────────────────────────────────────────────────────────────────
  0.000s  AudioChunk   shape=(8192, 8)
  0.000s  StftChunk    magnitudes=(8, 257, 33)  freqs=0–22050 Hz  frames=33
  0.000s  DoaChunk     azimuth=0.0°
  0.186s  AudioChunk   shape=(8192, 8)
  0.186s  StftChunk    magnitudes=(8, 257, 34)  freqs=0–22050 Hz  frames=34
  0.186s  DoaChunk     azimuth=315.0°
  0.372s  AudioChunk   shape=(8192, 8)
  0.372s  StftChunk    magnitudes=(8, 257, 34)  freqs=0–22050 Hz  frames=34
  0.372s  DoaChunk     azimuth=315.0°
  0.557s  AudioChunk   shape=(8192, 8)
  0.557s  StftChunk    magnitudes=(8, 257, 34)  freqs=0–22050 Hz  frames=34
  0.557s  DoaChunk     azimuth=90.0°
  0.743s  AudioChunk   shape=(8192, 8)
  0.743s  StftChunk    magnitudes=(8, 257, 34)  freqs=0–22050 Hz  frames=34
  0.743s  DoaChunk     azimuth=90.0°
  0.929s  AudioChunk   shape=(6656, 8)
  0.929s  StftChunk    magni